# 01 — Verify API Auth
**Day 1 | Part 7.3**

Tests the full VoltGrid API authentication flow end to end:
1. Reads credentials from Key Vault (never hardcoded)
2. Calls `POST /api/auth/login/` → gets a token at runtime
3. Uses the token to fetch data from the payments endpoint
4. Scans all 18 endpoints and confirms row counts
5. Runs a noise check on payment records

**Prerequisites:**
- `00b_connect_storage_no_mount` notebook already ran successfully (Cells 1–3)
- Key Vault secrets: `voltgrid-api-base-url`, `voltgrid-username`, `voltgrid-password`

## Cell 1 — Load API credentials from Key Vault

In [ ]:
import requests

SCOPE = "kv-ev-scope"

try:
    api_base_url = dbutils.secrets.get(scope=SCOPE, key="voltgrid-api-base-url")
    username     = dbutils.secrets.get(scope=SCOPE, key="voltgrid-username")
    password     = dbutils.secrets.get(scope=SCOPE, key="voltgrid-password")
    print(f"API base URL : {api_base_url}")
    print(f"Username     : {username}")
    print(f"Password     : [REDACTED]")
    print("Credentials loaded from Key Vault — OK")
except Exception as e:
    print(f"ERROR: {e}")
    print("Check that these secrets exist in Key Vault:")
    print("  voltgrid-api-base-url, voltgrid-username, voltgrid-password")
    raise

## Cell 2 — Login and get token at runtime

In [ ]:
try:
    resp = requests.post(
        f"{api_base_url}/api/auth/login/",
        json={"username": username, "password": password},
        timeout=10,
    )
    resp.raise_for_status()
    token = resp.json()["token"]

    print(f"Login response status : {resp.status_code}")
    print(f"Token acquired        : {token[:8]}...[REDACTED]")
    print("API login — OK")

    API_TOKEN   = token
    API_HEADERS = {"Authorization": f"Token {API_TOKEN}"}

except requests.exceptions.HTTPError as e:
    print(f"HTTP ERROR {resp.status_code}: {resp.text}")
    if resp.status_code == 401:
        print("→ Username or password in Key Vault does not match the API.")
        print("  Check voltgrid-username and voltgrid-password secrets in Key Vault.")
    raise
except requests.exceptions.ConnectionError:
    print(f"CONNECTION ERROR — cannot reach {api_base_url}")
    print("→ Check voltgrid-api-base-url secret value in Key Vault.")
    raise
except Exception as e:
    print(f"ERROR: {e}")
    raise

## Cell 3 — Fetch first page of payments endpoint

In [ ]:
try:
    r = requests.get(
        f"{api_base_url}/api/db/payments/?page=1&page_size=5",
        headers=API_HEADERS,
        timeout=10,
    )
    r.raise_for_status()
    data = r.json()

    pg = data.get("pagination", {})
    print(f"Payments endpoint response:")
    print(f"  Total records        : {pg.get('total', 'N/A'):,}")
    print(f"  Total pages          : {pg.get('total_pages', 'N/A'):,}")
    print(f"  Page size            : {pg.get('page_size', 'N/A')}")
    print(f"  Records in this page : {len(data.get('data', []))}")

    print(f"\nSample record:")
    records = data.get("data", [])
    if records:
        for k, v in records[0].items():
            print(f"  {k:<25} : {v}")

    print("\nPayments API call — OK")

except requests.exceptions.HTTPError as e:
    print(f"HTTP ERROR {r.status_code}: {r.text}")
    if r.status_code == 401:
        print("→ Token may have expired. Re-run Cell 2 to get a fresh token.")
    raise
except Exception as e:
    print(f"ERROR: {e}")
    raise

## Cell 4 — Scan all 18 API endpoints

In [ ]:
ENDPOINTS = [
    "payments", "sessions", "customers", "fleet", "chargers",
    "vehicles", "stations", "complaints", "maintenance_events",
    "energy_prices", "tariffs", "charge_cards", "employees",
    "partners", "cities", "states", "weather", "pipeline_audit"
]

print(f"{'Endpoint':<25} {'Status':>8} {'Total Rows':>12} {'Total Pages':>13}")
print("-" * 65)

endpoint_errors = []
for ep in ENDPOINTS:
    try:
        r = requests.get(
            f"{api_base_url}/api/db/{ep}/?page=1&page_size=1",
            headers=API_HEADERS,
            timeout=10,
        )
        if r.status_code == 200:
            pg    = r.json().get("pagination", {})
            total = pg.get("total", 0)
            pages = pg.get("total_pages", 0)
            print(f"{ep:<25} {'200 OK':>8} {total:>12,} {pages:>13,}")
        else:
            print(f"{ep:<25} {r.status_code:>8} {'ERROR':>12}")
            endpoint_errors.append(ep)
    except Exception as e:
        print(f"{ep:<25} {'FAIL':>8} {str(e)[:30]:>12}")
        endpoint_errors.append(ep)

print("-" * 65)
if endpoint_errors:
    print(f"\nEndpoints with errors: {endpoint_errors}")
    print("Re-run Cell 2 to refresh the token if you see 401 errors.")
else:
    print(f"\nAll {len(ENDPOINTS)} endpoints reachable — API auth verified.")

## Cell 5 — Noise check on payments

In [ ]:
print("Fetching 500 payment records for noise check...")

r = requests.get(
    f"{api_base_url}/api/db/payments/?page=1&page_size=500",
    headers=API_HEADERS,
    timeout=30,
)
r.raise_for_status()
recs = r.json().get("data", [])

if not recs:
    print("ERROR: No records returned from payments endpoint.")
    print("  → Check the raw response keys: print(r.json().keys())")
    print("  → Re-run Cell 2 to get a fresh token and try again.")
else:
    VALID_STATUS = {"Success", "Failed", "Pending", "Retry", "Refunded", "Disputed"}

    neg_amount  = [x for x in recs if float(x.get("amount_aud", 0) or 0) < 0]
    zero_amount = [x for x in recs if float(x.get("amount_aud", 0) or 0) == 0]
    bad_status  = [x for x in recs if x.get("status", "") not in VALID_STATUS]

    total = len(recs)
    print(f"\nNoise check on {total} payment records:")
    print(f"  Negative amounts  : {len(neg_amount):>5} ({len(neg_amount)/total*100:.1f}%) — expected ~5%")
    print(f"  Zero amounts      : {len(zero_amount):>5} ({len(zero_amount)/total*100:.1f}%) — expected ~5%")
    print(f"  Invalid status    : {len(bad_status):>5} ({len(bad_status)/total*100:.1f}%) — expected ~5%")

    if neg_amount:
        s = neg_amount[0]
        print(f"\n  Sample negative: payment_id={s.get('payment_id')}, amount={s.get('amount_aud')}, status={s.get('status')}")

    if bad_status:
        s = bad_status[0]
        print(f"  Sample bad status: payment_id={s.get('payment_id')}, status='{s.get('status')}'")

    print("\nNoise check complete — Silver layer will clean these in Day 7.")